# Artists vs. Diffusion Models — Experiment 2: Caption Generation & CLIP

Generates images from BLIP-2 captions for Rutkowski and Bowater using SD 1.5, then measures CLIP similarity between originals and generations.

Self-contained. **Hardware:** T4 GPU. **Runtime:** ~45 min.

---
## Setup

In [39]:
# Colab Setup
import os, sys

os.environ['TRANSFORMERS_NO_TF'] = '1'

ON_COLAB = 'google.colab' in str(get_ipython())
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PATH = "/content/drive/Othercomputers/laptop/research/diffusion_art"
else:
    for p in ["/mnt/c/research/diffusion_art", os.path.expanduser("~/research/diffusion_art"), "."]:
        if os.path.exists(os.path.join(p, "assets")):
            PATH = os.path.abspath(p); break

print(f"ON_COLAB: {ON_COLAB}")
print(f"PATH: {PATH}")
os.chdir(PATH)
sys.path.insert(0, PATH)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ON_COLAB: True
PATH: /content/drive/Othercomputers/laptop/research/diffusion_art


In [40]:
# Install dependencies
!pip install -q --upgrade diffusers transformers accelerate
!pip install -q torch torchvision pillow
!pip install -q open_clip_torch ftfy regex
!pip install -q plotly

if not os.path.exists("csd_repo"):
    !git clone -q https://github.com/learn2phoenix/CSD.git csd_repo
sys.path.insert(0, os.path.join(PATH, "csd_repo"))

print("Dependencies installed.")
print("IMPORTANT: Runtime → Restart session, then run all cells from the top.")


Dependencies installed.
IMPORTANT: Runtime → Restart session, then run all cells from the top.


In [41]:
import tensorflow as tf  # must be first to prevent proto conflicts
import os, sys, json, glob
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio

ASSETS_DIR    = os.path.join(PATH, "assets")
PORTFOLIO_DIR = os.path.join(ASSETS_DIR, "portfolio")
CONTROL_DIR   = os.path.join(ASSETS_DIR, "control")
GENERATED_DIR = os.path.join(ASSETS_DIR, "generated")
FIGURES_DIR   = os.path.join(ASSETS_DIR, "figures")
RESULTS_DIR   = os.path.join(PATH, "results")

for d in [FIGURES_DIR, RESULTS_DIR,
          os.path.join(GENERATED_DIR, "sd15_style"),
          os.path.join(GENERATED_DIR, "sd21_style"),
          os.path.join(GENERATED_DIR, "sd15_caption")]:
    os.makedirs(d, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

results = {}
pio.templates.default = "plotly_white"

def save_fig(fig, path):
    base = path.replace(".png", "")
    pio.write_json(fig, base + ".json")
    fig.write_html(base + ".html", include_plotlyjs="cdn")

print("Imports OK.")


Device: cuda
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB
Imports OK.


In [42]:
def load_images(directory, size=(512, 512)):
    """Load all images from a directory, return as list of PIL Images."""
    extensions = ["*.png", "*.jpg", "*.jpeg", "*.webp",
                  "*.PNG", "*.JPG", "*.JPEG", "*.WEBP"]
    paths = []
    for ext in extensions:
        paths.extend(glob.glob(os.path.join(directory, ext)))
    paths = sorted(set(paths))  # deduplicate in case of case-insensitive FS

    images = []
    skipped = []
    for p in paths:
        try:
            img = Image.open(p).convert("RGB")
            if size:
                img = img.resize(size, Image.LANCZOS)
            images.append({"path": p, "name": os.path.basename(p), "image": img})
        except Exception as e:
            skipped.append((os.path.basename(p), str(e)))

    print(f"  Loaded {len(images)} images from {directory}")
    if skipped:
        print(f"  Skipped {len(skipped)} files:")
        for name, err in skipped:
            print(f"    {name}: {err}")
    return images

print("Loading Rutkowski portfolio...")
rutkowski_imgs = load_images(PORTFOLIO_DIR)

if not rutkowski_imgs:
    print("WARNING: No Rutkowski images found. Add images to assets/portfolio/")

Loading Rutkowski portfolio...
  Loaded 30 images from /content/drive/Othercomputers/laptop/research/diffusion_art/assets/portfolio


---
## Section 3b: Caption-Conditioned Generation

In [43]:
# Section 3b: Does "by Greg Rutkowski" shift generations toward his style?
# For each of 30 portfolio images: generate with and without the artist name,
# then measure CLIP similarity to the original. A gap means the name carries style signal.
from diffusers import StableDiffusionPipeline
from transformers import BlipProcessor, BlipForConditionalGeneration

MODEL_ID_15  = "runwayml/stable-diffusion-v1-5"
SEED         = 42
NUM_STEPS    = 30
GUIDANCE     = 7.5
save_dir_cap = os.path.join(GENERATED_DIR, "sd15_caption")
os.makedirs(save_dir_cap, exist_ok=True)

# ── BLIP captions ────────────────────────────────────────────────────────────
print("Loading BLIP...")
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-large", torch_dtype=torch.float16).to(device)

def blip_caption(img):
    inputs = blip_processor(img, text="a painting of",
                            return_tensors="pt").to(device, torch.float16)
    ids = blip_model.generate(**inputs, max_length=80)
    return blip_processor.decode(ids[0], skip_special_tokens=True).strip()

print(f"Captioning {len(rutkowski_imgs)} images...")
captions = {}
for item in rutkowski_imgs:
    cap = blip_caption(item["image"])
    captions[item["name"]] = cap
    print(f"  {item['name']}: {cap[:70]}")

del blip_model, blip_processor
torch.cuda.empty_cache()

# ── Generate: with and without artist name ────────────────────────────────────
print("\nLoading SD 1.5...")
pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID_15, torch_dtype=torch.float16,
    safety_checker=None, requires_safety_checker=False).to(device)
pipe.enable_attention_slicing()

def generate(prompt, prefix):
    gen = torch.Generator(device=device).manual_seed(SEED)
    with torch.autocast("cuda"):
        img = pipe(prompt, num_inference_steps=NUM_STEPS,
                   guidance_scale=GUIDANCE, generator=gen).images[0]
    fname = f"{prefix}_seed{SEED}.png"
    img.save(os.path.join(save_dir_cap, fname))
    return img

without_name, with_name = [], []
for item in rutkowski_imgs:
    cap = captions[item["name"]]
    img_no   = generate(cap,                         f"{item['name']}_no_name")
    img_with = generate(cap + ", by Greg Rutkowski",  f"{item['name']}_with_name")
    without_name.append({"original": item["image"], "generated": img_no,   "caption": cap, "name": item["name"]})
    with_name.append(   {"original": item["image"], "generated": img_with, "caption": cap, "name": item["name"]})
    print(f"  {item['name'][:40]}: done")

del pipe
torch.cuda.empty_cache()
print(f"\nGenerated {len(without_name)} pairs (with and without artist name).")


Loading BLIP...


Loading weights:   0%|          | 0/616 [00:00<?, ?it/s]

Captioning 30 images...
  greg-rutkowski-after-battle-final-last.jpg: a painting of a group of men in armor holding flags
  greg-rutkowski-avatar-of-growth.jpg: a painting of a giant creature in a forest with trees
  greg-rutkowski-castle-defence-1920.jpg: a painting of a dragon flying over a castle with a crowd of people
  greg-rutkowski-demon-berserker-1200.jpg: a painting of a demonic looking man holding a large axe
  greg-rutkowski-dragonslayer-key-art-final-1920.jpg: a painting of a man holding a sword in front of a dragon
  greg-rutkowski-enhanced-surveillance-1500.jpg: a painting of a man in a hooded jacket standing in a street
  greg-rutkowski-firemind-s-research.jpg: a painting of a dragon with a lightning bolt coming out of its mouth
  greg-rutkowski-fisherman-boy-1500.jpg: a painting of a man sitting on a dock with a fishing rod
  greg-rutkowski-ghosts-of-saltmarsh-1920.jpg: a painting of a boat with people on it in a stormy sea
  greg-rutkowski-god-eternal-oketra-1600.jpg: 

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

...

  0%|          | 0/30 [00:00<?, ?it/s]

  greg-rutkowski-vraska-scheming-gorgon-12: done


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  greg-rutkowski-winter-river-study-1500.j: done

Generated 30 pairs (with and without artist name).


In [44]:
# CLIP similarity: original vs. generation, with vs. without artist name
import open_clip

print("Loading CLIP ViT-L/14...")
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-L-14", pretrained="openai")
clip_model = clip_model.to(device).eval()

def clip_sim(img1, img2):
    def emb(img):
        t = clip_preprocess(img).unsqueeze(0).to(device)
        with torch.no_grad():
            e = clip_model.encode_image(t)
            return (e / e.norm(dim=-1, keepdim=True)).cpu().float()
    return (emb(img1) * emb(img2)).sum().item()

sims_no   = [clip_sim(r["original"], r["generated"]) for r in without_name]
sims_with = [clip_sim(r["original"], r["generated"]) for r in with_name]

results["sec3b"] = {
    "clip_mean_without_name": float(np.mean(sims_no)),
    "clip_std_without_name":  float(np.std(sims_no)),
    "clip_mean_with_name":    float(np.mean(sims_with)),
    "clip_std_with_name":     float(np.std(sims_with)),
    "n": len(sims_no),
}
print(f"Without name: {np.mean(sims_no):.3f} ± {np.std(sims_no):.3f}")
print(f"With name:    {np.mean(sims_with):.3f} ± {np.std(sims_with):.3f}")
print(f"Gap:          {np.mean(sims_with) - np.mean(sims_no):+.3f}")


Loading CLIP ViT-L/14...
Without name: 0.704 ± 0.065
With name:    0.777 ± 0.045
Gap:          +0.073


In [45]:
# Figure 3a: Original | no-name generation | with-name generation (first 5 examples)
import gc; gc.collect()

# Pick the 5 examples with the largest gap (with_name - without_name)
gaps = [clip_sim(r["original"], w["generated"]) - clip_sim(r["original"], r["generated"])
        for r, w in zip(without_name, with_name)]
top_idx = sorted(range(len(gaps)), key=lambda i: gaps[i], reverse=True)[:5]
sel_no   = [without_name[i] for i in top_idx]
sel_with = [with_name[i]    for i in top_idx]
n = len(sel_no)
fig, axes = plt.subplots(n, 3, figsize=(12, n * 4))
fig.suptitle("Figure 3a: Caption only vs. Caption + \"by Greg Rutkowski\"",
             fontsize=13, fontweight="bold")

for i, (row_no, row_with) in enumerate(zip(sel_no, sel_with)):
    sim_no   = clip_sim(row_no["original"],   row_no["generated"])
    sim_with = clip_sim(row_with["original"], row_with["generated"])
    axes[i][0].imshow(row_no["original"])
    axes[i][0].set_title("Original", fontsize=9, fontweight="bold")
    axes[i][0].axis("off")
    axes[i][1].imshow(row_no["generated"])
    axes[i][1].set_title(f"Caption only\nCLIP={sim_no:.3f}", fontsize=8)
    axes[i][1].axis("off")
    axes[i][2].imshow(row_with["generated"])
    axes[i][2].set_title(f"+ by Greg Rutkowski\nCLIP={sim_with:.3f}", fontsize=8, color="#dc2626")
    axes[i][2].axis("off")
    cap_short = row_no["caption"][:55] + "..." if len(row_no["caption"]) > 55 else row_no["caption"]
    axes[i][0].set_xlabel(f'"{cap_short}"', fontsize=6, labelpad=3)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "fig3a_caption_gallery.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: fig3a_caption_gallery.png")


Output hidden; open in https://colab.research.google.com to view.

In [46]:
import gc
del clip_model
torch.cuda.empty_cache()
gc.collect()
print(f'GPU memory freed: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated')

GPU memory freed: 0.01 GB allocated


In [47]:
# Save all numerical results to results.json
import json

# Convert numpy types for JSON serialization
def convert(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    if isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    return obj

def deep_convert(obj):
    if isinstance(obj, dict):
        return {k: deep_convert(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [deep_convert(v) for v in obj]
    return convert(obj)

results_clean = deep_convert(results)
results_path = os.path.join(RESULTS_DIR, "results.json")

with open(results_path, "w") as f:
    json.dump(results_clean, f, indent=2)

print(f"Results saved to: {results_path}")
print(json.dumps(results_clean, indent=2))

Results saved to: /content/drive/Othercomputers/laptop/research/diffusion_art/results/results.json
{
  "sec3b": {
    "clip_mean_without_name": 0.7043738385041555,
    "clip_std_without_name": 0.06490907964206989,
    "clip_mean_with_name": 0.7774020612239838,
    "clip_std_with_name": 0.04484185502999227,
    "n": 30
  }
}
